# IBM AML: NoGNN: Tree giving tough competition to Benchmark GNNs (HI-Small, LI-Small, HI-Medium)

A small, readable recipe for the IBM synthetic anti-money-laundering data (Altman et al., NeurIPS 2023).
One tree model (scikit-learn `HistGradientBoostingClassifier`) on **29 inputs**: 9 basic transaction columns plus **20 account-history features**
(has this sender paid this account before? how fast is the sender adding new payees?).

Most of the notebook uses `HI-Small`. Section 3b also tests the same model on `LI-Small` (the lower-laundering version), and section 8 applies the `HI-Small` model to `HI-Medium` without retraining. Section 4 adds retroactive flagging, which raises F1 on all three datasets.

**Protocol:** a 60/20/20 split by time (like the IBM paper), decision threshold picked on validation only, score = F1 of the laundering class on test. The model, features and handling of simultaneous transactions are our own and differ from the paper's baselines.

**Run times:** every code cell prints how long it took, in a line starting with a stopwatch symbol. The times saved in this copy were measured on a 16 GB RAM; Kaggle's machine will give different numbers. The whole notebook takes about 10 minutes here (the last cell prints the exact total), most of it in loading the data, training in section 3, and sections 3b and 8.

In [ ]:
import time
from IPython import get_ipython

# Run-time stamp: after every cell finishes, print how long that cell took.
_NOTEBOOK_START = time.time()

def _fmt_time(s):
    if s < 1: return "under 1 second"
    if s < 60: return f"{s:.0f} seconds"
    m, sec = divmod(round(s), 60)
    return f"{m} min {sec:02d} s"

_state = {"t0": time.time()}
_ip = get_ipython()
if _ip is not None:
    _ip.events.register("pre_run_cell", lambda info: _state.update(t0=time.time()))
    _ip.events.register("post_run_cell", lambda result: print(f"\n⏱  This cell took {_fmt_time(time.time() - _state['t0'])} to run"))

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve

for d in [Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"), Path("data")]:
    if (d / "HI-Small_Trans.csv").exists():
        DATA = d
        break
else:
    raise FileNotFoundError("add the IBM AML dataset to the notebook (Add data -> ealtman2019/ibm-transactions-for-anti-money-laundering-aml)")
VARIANT = "HI-Small"   # used by sections 1-2 and 4-7, 9, 10; LI-Small and HI-Medium are run separately in sections 3b and 8
N_SEEDS = 3
print(DATA)

## 1. The 20 features

Every feature looks only backwards in time: it uses rows listed earlier in the file.

In [ ]:
import numpy as np
import pandas as pd

COLS = ["ts", "from_bank", "from_acct", "to_bank", "to_acct",
        "amt_recv", "cur_recv", "amt_paid", "cur_paid", "fmt", "label"]
FEATURE_WINDOWS = {"1h": 3600, "24h": 86400, "7d": 7 * 86400}


def load(path):
    tx = pd.read_csv(path, header=0, names=COLS, parse_dates=["ts"],
                     dtype={"from_bank": str, "to_bank": str})
    return tx.sort_values("ts", kind="stable").reset_index(drop=True)


def _window_counts(group_codes, secs, flag, window):
    """For each row: sum of `flag` over earlier rows of the same group within `window` seconds."""
    order = np.lexsort((secs, group_codes))
    key = group_codes[order].astype(np.int64) * 10**8 + secs[order]
    prefix = np.concatenate([[0], np.cumsum(flag[order])])
    idx = np.arange(len(key))
    left = np.searchsorted(key, key - window, side="left")
    out = np.empty(len(key), dtype=np.int64)
    out[order] = prefix[idx] - prefix[left]
    return out


def add_features(tx):
    df = tx.copy()
    secs = ((df.ts - df.ts.min()).dt.total_seconds()).to_numpy(np.int64)
    sender = pd.factorize(df.from_acct)[0]
    recv = pd.factorize(df.to_acct)[0]

    # pair history: has this sender paid this beneficiary before?
    pair = df.groupby(["from_acct", "to_acct"], sort=False)
    df["pair_prev_cnt"] = pair.cumcount()
    df["is_first_pair"] = (df.pair_prev_cnt == 0).astype(np.int8)
    df["pair_age_days"] = (df.ts - pair.ts.transform("min")).dt.total_seconds() / 86400
    df["hrs_since_last_pair"] = pair.ts.diff().dt.total_seconds() / 3600

    # sender history
    s = df.groupby("from_acct", sort=False)
    df["sender_prev_cnt"] = s.cumcount()
    df["sender_age_days"] = (df.ts - s.ts.transform("min")).dt.total_seconds() / 86400
    df["hrs_since_last_sender_tx"] = s.ts.diff().dt.total_seconds() / 3600
    df["sender_prev_partners"] = s.is_first_pair.cumsum() - df.is_first_pair

    # beneficiary history: first time this account receives anything
    r = df.groupby("to_acct", sort=False)
    df["recv_prev_cnt"] = r.cumcount()
    df["recv_age_days"] = (df.ts - r.ts.transform("min")).dt.total_seconds() / 86400

    # velocity: sender tx count and new-beneficiary count in trailing windows
    ones = np.ones(len(df), dtype=np.int64)
    first = df.is_first_pair.to_numpy(np.int64)
    for name, w in FEATURE_WINDOWS.items():
        df[f"sender_tx_{name}"] = _window_counts(sender, secs, ones, w)
        df[f"sender_new_ben_{name}"] = _window_counts(sender, secs, first, w)
        df[f"recv_tx_{name}"] = _window_counts(recv, secs, ones, w)
    df["new_ben_share_24h"] = df.sender_new_ben_24h / df.sender_tx_24h.clip(lower=1)
    return df


NEW_FEATURES = ["is_first_pair", "pair_prev_cnt", "pair_age_days", "hrs_since_last_pair",
                "sender_prev_cnt", "sender_age_days", "hrs_since_last_sender_tx",
                "sender_prev_partners", "recv_prev_cnt", "recv_age_days",
                "new_ben_share_24h"] + [f"{p}_{n}" for n in FEATURE_WINDOWS
                                         for p in ("sender_tx", "sender_new_ben", "recv_tx")]


In [ ]:
print(len(NEW_FEATURES), "history features:")
for f in NEW_FEATURES: print("  ", f)

## 2. Load, build features, split by time

In [ ]:
tx = load(DATA / f"{VARIANT}_Trans.csv")
print(f"{len(tx):,} transactions, {tx.label.sum():,} laundering ({tx.label.mean():.3%})")

BASIC = ["hour", "dow", "log_amt_paid", "cur_recv", "cur_paid", "fmt", "cross_cur", "cross_bank", "self_xfer"]

def prepare():
    df = add_features(tx)
    df["hour"], df["dow"] = df.ts.dt.hour, df.ts.dt.dayofweek
    df["log_amt_paid"] = np.log1p(df.amt_paid)
    df["cross_cur"] = (df.cur_recv != df.cur_paid).astype(np.int8)
    df["cross_bank"] = (df.from_bank != df.to_bank).astype(np.int8)
    df["self_xfer"] = (df.from_acct == df.to_acct).astype(np.int8)
    for c in ["cur_recv", "cur_paid", "fmt"]:
        df[c] = df[c].astype("category")
    n = len(df)
    c1, c2 = int(.6 * n), int(.8 * n)
    return df.iloc[:c1], df.iloc[c1:c2], df.iloc[c2:]

### A quick look at the data

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
GREY, BLUE, ORANGE = "#9aa0a6", "#1f77b4", "#e8710a"

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
rate = tx.groupby("fmt").label.agg(["mean", "sum", "size"]).sort_values("mean", ascending=False)
rate["mean"].plot.bar(ax=ax[0], title="Laundering rate by payment format")
hourly = tx.groupby(tx.ts.dt.hour).label.mean()
hourly.plot(ax=ax[1], title="Laundering rate by hour of day")
for lab, g in tx.groupby("label"):
    np.log10(g.amt_paid.clip(lower=0.01)).plot.hist(ax=ax[2], bins=60, alpha=.6, density=True,
                                                     label="laundering" if lab else "normal")
ax[2].set_title("log10(amount paid)"); ax[2].legend()
plt.tight_layout(); plt.show()
rate

## 3. Train and evaluate (threshold from validation, F1 of the laundering class on test)

In [ ]:
def best_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    return th[(2 * pr * rc / np.clip(pr + rc, 1e-12, None))[:-1].argmax()]

preds = {}       # name -> (test frame, test scores, threshold) for seed 0, used by the charts
all_preds = {}     # name -> [(test scores, threshold, validation scores) for each seed]
val_preds = {}   # name -> (validation frame, validation scores), for choosing settings without touching test

def evaluate(features, seeds=N_SEEDS, name=None):
    tr, va, te = prepare()
    rows = []
    for seed in range(seeds):
        m = HistGradientBoostingClassifier(max_iter=300, class_weight="balanced", categorical_features="from_dtype",
                                           early_stopping=True, validation_fraction=0.1, random_state=seed)
        m.fit(tr[features], tr.label)
        pv = m.predict_proba(va[features])[:, 1]
        thr = best_threshold(va.label, pv)
        pt, y = m.predict_proba(te[features])[:, 1], te.label.to_numpy()
        pred = pt >= thr
        tp = int((pred & (y == 1)).sum())
        p, r = tp / max(int(pred.sum()), 1), tp / max(int(y.sum()), 1)
        rows.append((100 * 2 * p * r / max(p + r, 1e-12), 100 * p, 100 * r, average_precision_score(y, pt)))
        all_preds.setdefault(name, []).append((pt, thr, pv))
        if seed == 0:
            preds[name] = (te, pt, thr)
            val_preds[name] = (va, pv)
    return np.array(rows)

results = {}
for name, feats in [("9 basic columns only", BASIC), ("basic + 20 history features", BASIC + NEW_FEATURES)]:
    results[name] = evaluate(feats, name=name)
    a = results[name]
    print(f"{name:28s} F1 = {a[:,0].mean():5.2f} ± {a[:,0].std():.2f}   "
          f"P = {a[:,1].mean():4.1f}   R = {a[:,2].mean():4.1f}   PR-AUC = {a[:,3].mean():.3f}", flush=True)

## 3b. `LI-Small`: the same model on the lower-laundering dataset

`LI-Small` has about half as much laundering as `HI-Small` and a third as many schemes. It is run here, before the comparison table, so our row in the table can include it. Same model, same 20 features, same 60/20/20 time split and validation-chosen threshold; it also gets retroactive flagging (explained in section 4) with the look-back fixed at 1 day. This uses a leaner, 32-bit version of the feature code (also used in section 8 for the 32-million-row `HI-Medium` file); it reproduces the section 1 features exactly on `HI-Small`.

In [ ]:
import numpy as np
import pyarrow as pa
import pyarrow.compute as pcm
import pyarrow.csv as pc

LEAN_NAMES = ["ts", "from_bank", "from_acct", "to_bank", "to_acct", "amt_recv", "cur_recv", "amt_paid", "cur_paid", "fmt", "label"]
LEAN_BASIC = ["hour", "dow", "log_amt_paid", "cur_recv", "cur_paid", "fmt", "cross_cur", "cross_bank", "self_xfer"]
HISTORY = ["is_first_pair", "pair_prev_cnt", "pair_age_days", "hrs_since_last_pair", "sender_prev_cnt", "sender_age_days",
           "hrs_since_last_sender_tx", "sender_prev_partners", "recv_prev_cnt", "recv_age_days", "new_ben_share_24h",
           "sender_tx_1h", "sender_new_ben_1h", "recv_tx_1h", "sender_tx_24h", "sender_new_ben_24h", "recv_tx_24h",
           "sender_tx_7d", "sender_new_ben_7d", "recv_tx_7d"]
LEAN_FEATURES = LEAN_BASIC + HISTORY
CATEGORICAL = [LEAN_FEATURES.index(c) for c in ("cur_recv", "cur_paid", "fmt")]
LEAN_WINDOWS = {"1h": 3600, "24h": 86400, "7d": 604800}


def _grp(code):
    order = np.argsort(code, kind="stable")          # arrays are already time-sorted, so order inside a group stays chronological
    sc = code[order]
    b = np.empty(len(sc), bool); b[0] = True; b[1:] = sc[1:] != sc[:-1]
    start = np.flatnonzero(b)[np.cumsum(b) - 1]
    return order, start


def _scatter(order, v):
    out = np.empty_like(v); out[order] = v
    return out


def _group_history(code, ts, flag=None):
    """per row: earlier rows in the group, age since the group's first row, seconds since the previous row, earlier-flag count"""
    order, start = _grp(code)
    n = len(code); pos = np.arange(n) - start
    t = ts[order]
    gap = (t - np.r_[t[:1], t[:-1]]).astype(np.float64); gap[pos == 0] = np.nan
    res = {"cnt": _scatter(order, pos), "age": _scatter(order, t - t[start]), "gap": _scatter(order, gap)}
    if flag is not None:
        f = flag[order]; c = np.cumsum(f)
        res["flag_before"] = _scatter(order, c - (c[start] - f[start]) - f)
    return res


def _lean_window_counts(group, secs, flags, windows):
    order = np.lexsort((secs, group))
    key = group[order] * 10**8 + secs[order]
    pre = {k: np.concatenate([[0], np.cumsum(v[order])]) for k, v in flags.items()}
    idx = np.arange(len(key)); out = {}
    for wn, w in windows.items():
        left = np.searchsorted(key, key - w, side="left")
        for k, p in pre.items():
            r = np.empty(len(key), np.int64); r[order] = p[idx] - p[left]
            out[(k, wn)] = r
    return out


def lean_read(path, fmts=None, curs=None):
    t = pc.read_csv(path, read_options=pc.ReadOptions(column_names=LEAN_NAMES, skip_rows=1),
                    convert_options=pc.ConvertOptions(column_types={
                        "ts": pa.string(), "from_bank": pa.string(), "from_acct": pa.string(), "to_bank": pa.string(), "to_acct": pa.string(),
                        "amt_recv": pa.float64(), "cur_recv": pa.string(), "amt_paid": pa.float64(), "cur_paid": pa.string(),
                        "fmt": pa.string(), "label": pa.int8()}))
    n = t.num_rows
    ts = pcm.strptime(t["ts"], format="%Y/%m/%d %H:%M", unit="s").cast(pa.int64()).to_numpy()
    both = pa.chunked_array(t["from_acct"].chunks + t["to_acct"].chunks)
    acct = pcm.index_in(both, value_set=pcm.unique(both)).to_numpy().astype(np.int64)
    del both
    fmts = fmts or sorted(pcm.unique(t["fmt"]).to_pylist())
    curs = curs or sorted(set(pcm.unique(t["cur_recv"]).to_pylist()) | set(pcm.unique(t["cur_paid"]).to_pylist()))
    code = lambda col, vals: pcm.fill_null(pcm.index_in(t[col], value_set=pa.array(vals)), -1).to_numpy().astype(np.int16)  # noqa: E731
    d = {"ts": ts, "sender": acct[:n], "recv": acct[n:], "amt_paid": t["amt_paid"].to_numpy(), "label": t["label"].to_numpy(),
         "fmt": code("fmt", fmts), "cur_recv": code("cur_recv", curs), "cur_paid": code("cur_paid", curs),
         "cross_cur": pcm.not_equal(t["cur_recv"], t["cur_paid"]).to_numpy(),
         "cross_bank": pcm.not_equal(t["from_bank"], t["to_bank"]).to_numpy()}
    del t
    order = np.argsort(d["ts"], kind="stable")
    return {k: v[order] for k, v in d.items()}, fmts, curs


def lean_features(d):
    """-> dict name -> float32 array for every name in LEAN_FEATURES, in time order"""
    ts, s, r = d["ts"], d["sender"], d["recv"]
    secs = ts - ts.min()
    A = int(max(s.max(), r.max())) + 1
    pair = _group_history(s * A + r, ts)
    first = (pair["cnt"] == 0).astype(np.int64)
    snd = _group_history(s, ts, first)
    rcv = _group_history(r, ts)
    out = {"hour": (ts // 3600) % 24, "dow": (ts // 86400 + 3) % 7, "log_amt_paid": np.log1p(d["amt_paid"]),
           "cur_recv": d["cur_recv"], "cur_paid": d["cur_paid"], "fmt": d["fmt"], "cross_cur": d["cross_cur"],
           "cross_bank": d["cross_bank"], "self_xfer": s == r,
           "is_first_pair": first, "pair_prev_cnt": pair["cnt"], "pair_age_days": pair["age"] / 86400, "hrs_since_last_pair": pair["gap"] / 3600,
           "sender_prev_cnt": snd["cnt"], "sender_age_days": snd["age"] / 86400, "hrs_since_last_sender_tx": snd["gap"] / 3600,
           "sender_prev_partners": snd["flag_before"], "recv_prev_cnt": rcv["cnt"], "recv_age_days": rcv["age"] / 86400}
    ones = np.ones(len(ts), np.int64)
    sw = _lean_window_counts(s, secs, {"tx": ones, "new_ben": first}, LEAN_WINDOWS)
    rw = _lean_window_counts(r, secs, {"tx": ones}, LEAN_WINDOWS)
    for wn in LEAN_WINDOWS:
        out[f"sender_tx_{wn}"], out[f"sender_new_ben_{wn}"], out[f"recv_tx_{wn}"] = sw[("tx", wn)], sw[("new_ben", wn)], rw[("tx", wn)]
    out["new_ben_share_24h"] = out["sender_new_ben_24h"] / np.maximum(out["sender_tx_24h"], 1)
    return {k: np.asarray(out[k]).astype(np.float32) for k in LEAN_FEATURES}


In [ ]:
def f1_p_r(y, flag):
    tp = int((flag & (y == 1)).sum()); p, r = tp / max(int(flag.sum()), 1), tp / max(int(y.sum()), 1)
    return 100 * 2 * p * r / max(p + r, 1e-12), 100 * p, 100 * r, tp, int(flag.sum()) - tp

LOWS = [1.0, 0.95, 0.8, 0.5, 0.2, 0.05, 0.01]          # earlier-score minimum for retroactive flagging; 1.0 = never (the baseline)
import gc
LEAN_PUBLISHED = {"HI-Small": 63.23, "LI-Small": 27.30, "HI-Medium": 65.70}   # IBM paper, GFP + XGBoost, trained on each dataset itself

def lean_prepare(name, fmts=None, curs=None):
    d, fmts, curs = lean_read(str(DATA / f"{name}_Trans.csv"), fmts, curs)
    return d, lean_features(d), fmts, curs

def lean_fit(f, y, seeds=N_SEEDS):
    n = len(y); c1, c2 = int(.6 * n), int(.8 * n)
    cols = lambda a, b: np.column_stack([f[c][a:b] for c in LEAN_FEATURES])
    Xtr, Xva, Xte = cols(0, c1), cols(c1, c2), cols(c2, n)
    out = []
    for seed in range(seeds):
        m = HistGradientBoostingClassifier(max_iter=300, class_weight="balanced", categorical_features=CATEGORICAL,
                                           early_stopping=True, validation_fraction=0.1, random_state=seed).fit(Xtr, y[:c1])
        pv = m.predict_proba(Xva)[:, 1]
        out.append((m, best_threshold(y[c1:c2], pv), pv, m.predict_proba(Xte)[:, 1]))
    return out

def retro_codes(ts, s_, r_, score, thr, window, low):      # same rule as retro_flag, on integer account codes
    secs = ts - ts.min(); flag = score >= thr
    ev = np.sort(np.concatenate([s_[flag] * 10**8 + secs[flag], r_[flag] * 10**8 + secs[flag]]))
    gap = np.full(len(ts), np.inf)
    for a in (s_, r_):
        pos = np.searchsorted(ev, a * 10**8 + secs + 1, side="left")
        ok = pos < len(ev); nxt = np.where(ok, ev[pos.clip(max=len(ev) - 1)], -1)
        gap = np.minimum(gap, np.where(ok & (nxt // 10**8 == a), nxt - a * 10**8 - secs, np.inf))
    return flag | ((gap <= window) & (score >= low))

def summarize(runs):
    a = np.array([r[:3] for r in runs], dtype=float)
    return {"F1": a[:, 0].mean(), "F1 std": a[:, 0].std(), "precision": a[:, 1].mean(), "recall": a[:, 2].mean()}

other = {}     # label -> summary dict

# --- LI-Small, own split, with and without retroactive flagging (look-back 1 day)
d, f, fm_s, cu_s = lean_prepare("LI-Small")
y = d["label"]; n = len(y); c1, c2 = int(.6 * n), int(.8 * n)
base_runs, retro_runs = [], []
for m, thr, pv, pt in lean_fit(f, y):
    base_runs.append(f1_p_r(y[c2:], pt >= thr))
    seg = lambda a, b: (d["ts"][a:b], d["sender"][a:b], d["recv"][a:b])      # noqa: E731
    low = max(LOWS, key=lambda lo: f1_p_r(y[c1:c2], retro_codes(*seg(c1, c2), pv, thr, 86400, lo))[0])
    retro_runs.append(f1_p_r(y[c2:], retro_codes(*seg(c2, n), pt, thr, 86400, low)))
other["LI-Small: model alone"] = summarize(base_runs)
other["LI-Small: + retroactive flagging (1 day)"] = summarize(retro_runs)
print(f"LI-Small: {n:,} rows, {int(y[c2:].sum())} laundering transactions in the test split")
del d, f; gc.collect()

pd.DataFrame(other).T.round(2)

## 4. Retroactive flagging: when an alert fires, look back

If the model flags a transaction, the earlier transactions of the same accounts are often the earlier steps of the same scheme, and on their own they looked ordinary. The rule: **also flag an earlier transaction if one of its two accounts is involved in a later flagged transaction within a look-back window, and the earlier transaction already scored above a minimum level.** No labels are used. The window and the minimum score are chosen on the validation split; the test split is scored once.

Two caveats. First, these are *delayed* alerts: the earlier transaction is flagged only after the later one fires (up to the window length), so this is retrospective tracing, not real-time prevention, and it isn't how the F1 scores in section 3 are computed. Second, the base predictions come from one seed of one model.

In [ ]:
T0 = tx.ts.min()

def retro_flag(part, score, thr, window, low):
    secs = ((part.ts - T0).dt.total_seconds()).to_numpy(np.int64)
    codes = pd.factorize(pd.concat([part.from_acct, part.to_acct], ignore_index=True))[0].astype(np.int64)
    fa, ta = codes[:len(part)], codes[len(part):]
    flag = score >= thr
    ev = np.sort(np.concatenate([fa[flag] * 10**8 + secs[flag], ta[flag] * 10**8 + secs[flag]]))   # (account, minute) of every alert
    gap = np.full(len(part), np.inf)
    for a in (fa, ta):
        pos = np.searchsorted(ev, a * 10**8 + secs + 1, side="left")        # this account's next alert in a later minute
        ok = pos < len(ev)
        nxt = np.where(ok, ev[pos.clip(max=len(ev) - 1)], -1)
        gap = np.minimum(gap, np.where(ok & (nxt // 10**8 == a), nxt - a * 10**8 - secs, np.inf))
    return flag | ((gap <= window) & (score >= low))

def f1_p_r(y, flag):
    tp = int((flag & (y == 1)).sum()); p, r = tp / max(int(flag.sum()), 1), tp / max(int(y.sum()), 1)
    return 100 * 2 * p * r / max(p + r, 1e-12), 100 * p, 100 * r, tp, int(flag.sum()) - tp

WINDOWS = {"1h": 3600, "6h": 6 * 3600, "24h": 86400, "3d": 3 * 86400, "7d": 7 * 86400}
LOWS = [1.0, 0.95, 0.8, 0.5, 0.2, 0.05, 0.01]          # 1.0 = never flag earlier transactions (the baseline)
KEY_M = "basic + 20 history features"
te, pt, thr = preds[KEY_M]; va, pv = val_preds[KEY_M]
grid = pd.DataFrame({wn: [f1_p_r(va.label.to_numpy(), retro_flag(va, pv, thr, W, lo))[0] for lo in LOWS] for wn, W in WINDOWS.items()},
                    index=[f">= {lo}" for lo in LOWS])
lo_i, wn = np.unravel_index(grid.to_numpy().argmax(), grid.shape); wn = grid.columns[wn]; lo = LOWS[lo_i]
before = f1_p_r(te.label.to_numpy(), pt >= thr)
after = f1_p_r(te.label.to_numpy(), retro_flag(te, pt, thr, WINDOWS[wn], lo))
print(f"chosen on validation = look back {wn}, earlier score >= {lo} (validation F1 {grid.to_numpy().max():.1f}, baseline {grid.iloc[0, 0]:.1f})")
print(f"  test before: F1 {before[0]:.2f}  P {before[1]:.1f}  R {before[2]:.1f}  caught {before[3]}  false alarms {before[4]}")
print(f"  test after : F1 {after[0]:.2f}  P {after[1]:.1f}  R {after[2]:.1f}  caught {after[3]}  false alarms {after[4]}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3), gridspec_kw={"width_ratios": [1.1, 1]})
im = ax[0].imshow(grid.to_numpy(), cmap="viridis", aspect="auto")
ax[0].set_xticks(range(grid.shape[1])); ax[0].set_xticklabels(grid.columns); ax[0].set_yticks(range(grid.shape[0])); ax[0].set_yticklabels(grid.index)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax[0].text(j, i, f"{grid.iat[i, j]:.0f}", ha="center", va="center", color="white", fontsize=9)
ax[0].set_xlabel("look-back window"); ax[0].set_ylabel("earlier transaction must already score")
ax[0].set_title("Validation F1 for each setting (top row = no retro flagging)")
x = np.arange(3); w = 0.38
ax[1].bar(x - w/2, before[:3], w, label="model alone", color=GREY)
ax[1].bar(x + w/2, after[:3], w, label=f"+ retro flagging ({wn}, >= {lo})", color=BLUE)
for i in range(3):
    ax[1].text(i - w/2, before[i] + 1, f"{before[i]:.1f}", ha="center", fontsize=9); ax[1].text(i + w/2, after[i] + 1, f"{after[i]:.1f}", ha="center", fontsize=9)
ax[1].set_xticks(x); ax[1].set_xticklabels(["F1", "precision", "recall"]); ax[1].set_ylim(0, 125); ax[1].legend(frameon=False, loc="upper center")
ax[1].set_title(f"Test split: caught {before[3]} -> {after[3]}, false alarms {before[4]} -> {after[4]}")
plt.tight_layout(); plt.show()

In [ ]:
# The same procedure on every seed (settings re-chosen on validation for each seed), so it is comparable with the F1 scores in section 3
def tune_apply(seed):
    te, _, _ = preds[KEY_M]; va, _ = val_preds[KEY_M]
    pt_s, thr_s, pv_s = all_preds[KEY_M][seed]
    yv = va.label.to_numpy()
    _, wn, lo = max(((f1_p_r(yv, retro_flag(va, pv_s, thr_s, W, l))[0], n, l) for n, W in WINDOWS.items() for l in LOWS), key=lambda t: t[0])
    yt = te.label.to_numpy()
    return f1_p_r(yt, pt_s >= thr_s), f1_p_r(yt, retro_flag(te, pt_s, thr_s, WINDOWS[wn], lo)), (wn, lo)

runs = [tune_apply(sd) for sd in range(N_SEEDS)]
retro_hi_f1 = float(np.mean([r[1][0] for r in runs]))      # mean test F1 with retroactive flagging, used in the comparison table
summary = []
for label, idx in [("model alone", 0), ("+ retroactive flagging", 1)]:
    a = np.array([r[idx][:3] for r in runs])
    summary.append({"variant": label, "F1": f"{a[:, 0].mean():.2f} ± {a[:, 0].std():.2f}", "precision": f"{a[:, 1].mean():.1f}", "recall": f"{a[:, 2].mean():.1f}",
                    "settings chosen per seed": "" if idx == 0 else ", ".join(f"{w}/{l}" for *_, (w, l) in runs)})
pd.DataFrame(summary)

## 5. Comparison with published methods (minority-class F1, `HI-Small` and `LI-Small`)

Our two rows are the model alone and the same model with retroactive flagging (section 4), which flags earlier transactions after a later alert, so its alerts are delayed. Published numbers are copied from the papers (Altman et al., NeurIPS 2023, Table 2; Lin et al., ICAIF 2024, *FraudGT*, for the Multi-* rows). They are **not** recomputed here, and their splits and handling of same-minute transactions differ from ours (FraudGT uses a 64/19/17 split on `HI-Small`; the IBM graph models see a whole-period snapshot), so treat the comparison as indicative, not a leaderboard.

In [ ]:
PUBLISHED = [  # (method, family, HI-Small F1, LI-Small F1, source)
    ("GIN",                          "graph neural net", 28.70,  7.90, "IBM paper"),
    ("GIN + edge updates",           "graph neural net", 47.73, 20.62, "IBM paper"),
    ("PNA",                          "graph neural net", 56.77, 16.45, "IBM paper"),
    ("LightGBM + graph features",    "trees",            62.86, 20.83, "IBM paper"),
    ("XGBoost + graph features",     "trees",            63.23, 27.30, "IBM paper"),
    ("Multi-PNA",                    "graph neural net", 67.35, 35.39, "FraudGT paper"),
    ("Multi-FraudGT",                "graph transformer",75.81, 45.69, "FraudGT paper"),
]
ours = results["basic + 20 history features"][:, 0]
table = pd.DataFrame(PUBLISHED, columns=["method", "family", "HI-Small F1", "LI-Small F1", "source"])
table = pd.concat([table, pd.DataFrame([
    ("Ours: 20 features + trees", "trees", ours.mean(), other["LI-Small: model alone"]["F1"], "this notebook"),
    ("Ours: + retroactive flagging (delayed alerts)", "trees", retro_hi_f1, other["LI-Small: + retroactive flagging (1 day)"]["F1"], "this notebook (section 4; look-back 3 days on HI-Small, 1 day on LI-Small)"),
], columns=table.columns)], ignore_index=True).sort_values("HI-Small F1", ascending=False).reset_index(drop=True)
table.style.format({"HI-Small F1": "{:.1f}", "LI-Small F1": "{:.1f}"}, na_rep="n/a").hide(axis="index")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
GREY, BLUE, ORANGE = "#9aa0a6", "#1f77b4", "#e8710a"

# Chart 1: F1 on HI-Small and LI-Small, ours highlighted
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, col in zip(axes, ["HI-Small F1", "LI-Small F1"]):
    t = table.sort_values(col)
    ax.barh(t.method, t[col], color=[ORANGE if "retroactive" in m else BLUE if m.startswith("Ours") else GREY for m in t.method])
    for y, v in enumerate(t[col]):
        ax.text(v + 0.6, y, f"{v:.1f}", va="center")
    ax.set_xlabel(f"minority-class F1 on {col[:8]} (higher is better)"); ax.set_xlim(0, 90)
    ax.set_title(f"{col[:8]}: published methods vs 20 history features + trees")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2: what the history features add
labels = ["basic columns only", "+ 20 history features"]
keys = ["9 basic columns only", "basic + 20 history features"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
f1 = [results[k][:, 0].mean() for k in keys]; sd = [results[k][:, 0].std() for k in keys]
ax[0].bar(labels, f1, yerr=sd, color=[GREY, BLUE], capsize=4)
for i, v in enumerate(f1): ax[0].text(i, v + 2.5, f"{v:.1f}", ha="center")
ax[0].set_ylabel("F1 (mean ± std over seeds)"); ax[0].set_title("F1 on HI-Small test")
w = 0.38; x = np.arange(2)
ax[1].bar(x - w/2, [results[k][:, 1].mean() for k in keys], w, label="precision", color=BLUE)
ax[1].bar(x + w/2, [results[k][:, 2].mean() for k in keys], w, label="recall", color=ORANGE)
ax[1].set_xticks(x); ax[1].set_xticklabels(labels); ax[1].set_ylabel("%"); ax[1].legend(frameon=False)
ax[1].set_title("Precision and recall at the validation-chosen threshold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2b: how many laundering transactions did we actually catch? (test split, seed 0, validation-chosen threshold)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
names = ["basic columns only", "+ 20 history features"]
caught, missed, false_alarms = [], [], []
for k in keys:
    te, pt, thr = preds[k]
    y, flag = te.label.to_numpy() == 1, pt >= thr
    caught.append(int((y & flag).sum())); missed.append(int((y & ~flag).sum())); false_alarms.append(int((~y & flag).sum()))
total = caught[0] + missed[0]
ax[0].bar(names, caught, color=BLUE, label="caught")
ax[0].bar(names, missed, bottom=caught, color=GREY, label="missed")
for i, (c, m) in enumerate(zip(caught, missed)):
    ax[0].text(i, c / 2, f"{c:,}\n({c / (c + m):.0%})", ha="center", va="center", color="white", fontsize=10)
ax[0].set_ylabel("laundering transactions in test split"); ax[0].set_ylim(0, total * 1.12); ax[0].legend(frameon=False, loc="upper center", ncol=2)
ax[0].set_title(f"Laundering transactions caught vs missed (about {total:,} in test)")
ax[1].bar(names, false_alarms, color=ORANGE)
for i, (f, c) in enumerate(zip(false_alarms, caught)):
    ax[1].text(i, f, f"{f:,}\n(precision {c / max(c + f, 1):.0%})", ha="center", va="bottom", fontsize=10)
ax[1].set_ylim(0, max(false_alarms) * 1.25); ax[1].set_ylabel("normal transactions flagged")
ax[1].set_title("False alarms (of about 1,016,000 normal)")
plt.tight_layout(); plt.show()
print(pd.DataFrame({"caught": caught, "missed": missed, "false alarms": false_alarms}, index=[n.replace("\n", " ") for n in names]))

In [ ]:
# Chart 3: precision-recall curves on the test split
fig, ax = plt.subplots(figsize=(6.5, 5))
for key, color, ls in [("basic + 20 history features", BLUE, "-"), ("9 basic columns only", GREY, "--")]:
    te, pt, thr = preds[key]
    pr, rc, _ = precision_recall_curve(te.label, pt)
    ax.plot(rc[:-1], pr[:-1], color=color, ls=ls, label=f"{'20 history features' if 'history' in key else 'basic columns only'} "
            f"(AP {average_precision_score(te.label, pt):.2f})")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02); ax.legend(frameon=False, loc="upper right")
ax.set_title("Precision-recall on the test split")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 4: where the features work and where they fail (test split)
te, pt, thr = preds["basic + 20 history features"]
te = te.assign(caught=(pt >= thr).astype(int))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

b = pd.cut(te.sender_new_ben_24h, [-1, 0, 1, 3, 10**9], labels=["0", "1", "2-3", "4+"])
r = te.groupby(b, observed=True).label.mean() * 100
ax[0].bar(r.index.astype(str), r.values, color=BLUE)
ax[0].set_xlabel("new payees the sender added in the previous 24h"); ax[0].set_ylabel("% of transactions that are laundering")
ax[0].set_title("Laundering rate by new-payee velocity")

pos = te[te.label == 1]
g = pos.groupby(pos.fmt.astype(str)).caught.agg(["mean", "sum", "size"]).sort_values("size", ascending=False)
ax[1].bar(g.index, g["mean"] * 100, color=BLUE)
for i, (m, c, n) in enumerate(zip(g["mean"], g["sum"], g["size"])): ax[1].text(i, m * 100 + 2, f"{c} of {n}", ha="center", fontsize=9)
ax[1].set_ylabel("recall %"); ax[1].set_ylim(0, 105); ax[1].set_title("Caught / total laundering, by payment format")
ax[1].tick_params(axis="x", rotation=30)

b = pd.cut(pos.sender_tx_24h, [-1, 0, 2, 10, 10**9], labels=["0", "1-2", "3-10", ">10"])
g = pos.groupby(b, observed=True).caught.agg(["mean", "sum", "size"])
ax[2].bar(g.index.astype(str), g["mean"] * 100, color=BLUE)
for i, (m, c, n) in enumerate(zip(g["mean"], g["sum"], g["size"])): ax[2].text(i, m * 100 + 2, f"{c} of {n}", ha="center", fontsize=9)
ax[2].set_xlabel("sender transactions in the previous 24h"); ax[2].set_ylabel("recall %"); ax[2].set_ylim(0, 105)
ax[2].set_title("Caught / total laundering, by sender activity")
plt.tight_layout(); plt.show()

## 6. What the labelled laundering schemes look like

`*_Patterns.txt` lists every laundering scheme the generator injected, grouped by type. Each graph below is one example (4 to 25 transactions). Single-transaction features can't see this structure, which is why account history helps.

In [ ]:
import re
import networkx as nx

pat_file = DATA / f"{VARIANT}_Patterns.txt"
if pat_file.exists():
    blocks = re.findall(r"BEGIN LAUNDERING ATTEMPT - ([^\n:]+).*?\n(.*?)END LAUNDERING ATTEMPT", pat_file.read_text(), flags=re.S)
    schemes = [(kind.strip(), pd.DataFrame([l.split(",") for l in body.strip().splitlines() if l.strip()], columns=COLS))
               for kind, body in blocks]
    print(len(schemes), "schemes;", pd.Series([k for k, _ in schemes]).value_counts().to_dict())

    def draw(kind, g, ax):
        G = nx.MultiDiGraph()
        for r in g.itertuples():
            G.add_edge(r.from_acct[-4:], r.to_acct[-4:])
        nx.draw(G, nx.spring_layout(G, seed=1), ax=ax, node_size=120, arrowsize=10, with_labels=False, node_color="#d62728")
        ax.set_title(f"{kind} ({G.number_of_nodes()} accounts, {G.number_of_edges()} txns)", fontsize=9)

    kinds = list(dict.fromkeys(k for k, _ in schemes))
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, kind in zip(axes.ravel(), kinds):
        draw(kind, next(g for k, g in schemes if k == kind and 4 <= len(g) <= 25), ax)
    for ax in axes.ravel()[len(kinds):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Patterns file not found; skipping scheme graphs.")

## 7. Which kinds of laundering scheme did we catch?

Each laundering transaction in the test split is matched to its scheme in `*_Patterns.txt` (by timestamp, accounts and payment format). Counts are laundering **transactions** from seed 0 at the validation-chosen threshold. Some laundering transactions aren't listed in any scheme in the patterns file; they get their own bar.

In [ ]:
GREEN, RED, LIGHT = "#2ca02c", "#d62728", "#c8c8c8"
if not pat_file.exists():
    print("Patterns file not found; skipping scheme analysis.")
else:
    pat = pd.concat([g.assign(scheme_id=i, scheme=k) for i, (k, g) in enumerate(schemes)], ignore_index=True)
    pat["ts"] = pd.to_datetime(pat.ts)
    pat = pat.rename(columns={"fmt": "fmt_raw"})
    KEY = ["ts", "from_acct", "to_acct", "fmt_raw"]
    pat_u = pat.drop_duplicates(KEY)

    def test_positives(mode_key):
        te, pt, thr = preds[mode_key]
        t = te[["ts", "from_acct", "to_acct", "label"]].assign(fmt_raw=te.fmt.astype(str), caught=(pt >= thr).astype(int))
        return t[t.label == 1].groupby(KEY, as_index=False).caught.max()

    def by_scheme(mode_key):
        t = test_positives(mode_key).merge(pat_u[KEY + ["scheme_id", "scheme"]], on=KEY, how="left")
        t["scheme"] = t.scheme.fillna("not in a listed scheme")
        c = t.groupby("scheme").caught.agg(caught="sum", total="size")
        c["missed"] = c.total - c.caught
        return c.sort_values("total", ascending=False)

    tab = by_scheme("basic + 20 history features")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(tab.index, tab.caught, color=GREEN, label="caught")
    ax.bar(tab.index, tab.missed, bottom=tab.caught, color=RED, label="missed")
    for i, (a, tot) in enumerate(zip(tab.caught, tab.total)):
        ax.text(i, tot + 6, f"{int(a)}/{int(tot)}\n({a / tot:.0%})", ha="center", fontsize=9)
    ax.set_title("Laundering transactions by scheme type"); ax.tick_params(axis="x", rotation=40)
    ax.set_ylim(0, tab.total.max() * 1.22); ax.set_ylabel("laundering transactions in the test split"); ax.legend(frameon=False, loc="upper right")
    plt.tight_layout(); plt.show()
    print(tab[["caught", "missed", "total"]])

Below, each transaction of an example scheme is coloured **green** if the model caught it and **red** if it missed it (grey = outside the test split). For each scheme type, the first set shows the example with the highest share caught and the second the example with the lowest.

In [ ]:
if pat_file.exists():
    t = test_positives("basic + 20 history features").merge(pat_u[KEY + ["scheme_id"]], on=KEY, how="inner")
    frame = pat_u.merge(t[KEY + ["caught"]], on=KEY, how="left")      # caught is NaN outside the test split
    stats = frame.groupby("scheme_id").agg(kind=("scheme", "first"), n=("scheme", "size"), n_test=("caught", "count"), n_caught=("caught", "sum"))
    stats["frac"] = stats.n_caught / stats.n_test.clip(lower=1)
    ok = stats[(stats.n >= 4) & (stats.n <= 25) & (stats.n_test >= np.maximum(3, 0.8 * stats.n))]

    def draw_outcome(sid, ax):
        sub = frame[frame.scheme_id == sid]
        G = nx.MultiDiGraph()
        for r in sub.itertuples():
            G.add_edge(r.from_acct[-4:], r.to_acct[-4:], c=LIGHT if pd.isna(r.caught) else GREEN if r.caught else RED)
        pos = nx.spring_layout(G, seed=1)
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=70, node_color="#444444")
        edges = list(G.edges(keys=True, data=True))
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=[(u, v, k) for u, v, k, _ in edges], edge_color=[d["c"] for *_, d in edges],
                               width=2, arrowsize=10, node_size=70)
        ax.set_axis_off()
        return int(stats.n_caught[sid]), int(stats.n_test[sid])

    kinds = list(dict.fromkeys(k for k, _ in schemes))
    for title, pick in [("highest share caught", lambda d: d.sort_values(["frac", "n"], ascending=[False, False]).index[0]),
                        ("lowest share caught", lambda d: d.sort_values(["frac", "n"], ascending=[True, False]).index[0])]:
        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        for ax, kind in zip(axes.ravel(), kinds):
            d = ok[ok.kind == kind]
            if d.empty:
                ax.set_axis_off(); ax.set_title(f"{kind}: no example fully in the test split", fontsize=9); continue
            c, n = draw_outcome(pick(d), ax)
            ax.set_title(f"{kind}: caught {c} of {n}", fontsize=10)
        fig.suptitle(f"Example scheme per type, {title} (green = caught, red = missed)", fontsize=13)
        plt.tight_layout(); plt.show()

## 8. `HI-Small` to `HI-Medium` without retraining

`HI-Medium` has 32 million transactions and is a separate, larger simulation (only about 2% of its account IDs also appear in `HI-Small`). The model trained on `HI-Small` is applied **directly**, with no retraining, alone and with retroactive flagging (1 day). The threshold, and the minimum earlier score for retroactive flagging, are the ones chosen on `HI-Small` validation; nothing from `HI-Medium` is used. The leaner feature code from section 3b is what lets the 32 million rows fit in memory; its `HI-Small` F1 is about 1 point different from section 3 (we believe because of the 32-bit storage, not checked further). This section takes about 4 minutes.

In [ ]:
# --- HI-Small, then HI-Medium scored directly with the HI-Small models
d, f, fm, cu = lean_prepare("HI-Small")
y = d["label"]; n = len(y); c2 = int(.8 * n)
fits = lean_fit(f, y)
other["HI-Small: own test split"] = summarize([f1_p_r(y[c2:], pt >= thr) for m, thr, pv, pt in fits])
c1_ = int(.6 * n)
sm_val, y_sm_val = (d["ts"][c1_:c2], d["sender"][c1_:c2], d["recv"][c1_:c2]), y[c1_:c2]
del d, f; gc.collect()

if (DATA / "HI-Medium_Trans.csv").exists():
    d2, f2, _, _ = lean_prepare("HI-Medium", fm, cu)
    y2 = d2["label"]; N = len(y2); k2 = int(.8 * N)
    ts2, s2, r2 = d2["ts"], d2["sender"], d2["recv"]
    last, whole, retro_a = [], [], []
    for m, thr, pv, pt in fits:
        sc = np.concatenate([m.predict_proba(np.column_stack([f2[c][a:a + 2_000_000] for c in LEAN_FEATURES]))[:, 1] for a in range(0, N, 2_000_000)])
        last.append(f1_p_r(y2[k2:], sc[k2:] >= thr)); whole.append(f1_p_r(y2, sc >= thr))
        low = max(LOWS, key=lambda lo: f1_p_r(y_sm_val, retro_codes(*sm_val, pv, thr, 86400, lo))[0])      # chosen on HI-Small validation only
        retro_a.append(f1_p_r(y2[k2:], retro_codes(ts2[k2:], s2[k2:], r2[k2:], sc[k2:], thr, 86400, low)))
    other["HI-Medium last 20%: HI-Small model, no retraining"] = summarize(last)
    other["HI-Medium last 20%: HI-Small model, + retroactive flagging (1 day)"] = summarize(retro_a)
    print(f"HI-Medium: {N:,} rows, {int(y2[k2:].sum()):,} laundering transactions in its last 20%")
    del d2, f2; gc.collect()
else:
    print("HI-Medium_Trans.csv not found; skipping the direct transfer test.")

pd.DataFrame(other).T.round(2)

In [ ]:
# Chart 5: our tree model on each dataset vs the published GFP + XGBoost (which is trained on each dataset itself)
groups = ["HI-Small", "LI-Small", "HI-Medium"]
ours = [float(results["basic + 20 history features"][:, 0].mean()), other["LI-Small: model alone"]["F1"],   # HI-Small: same numbers as the table above
        other.get("HI-Medium: HI-Small model, last 20%, no retraining", {"F1": np.nan})["F1"]]
retro_v = [retro_hi_f1, other["LI-Small: + retroactive flagging (1 day)"]["F1"],
           other.get("HI-Medium: HI-Small model, last 20%, + retroactive flagging (1 day)", {"F1": np.nan})["F1"]]
fig, ax = plt.subplots(figsize=(9, 4.3)); x = np.arange(3); w = 0.26
ax.bar(x - w, ours, w, label="ours (trees + 20 features); HI-Medium: trained on HI-Small", color=BLUE)
ax.bar(x, retro_v, w, label="ours + retroactive flagging (HI-Small: 3-day look-back, others: 1 day)", color=ORANGE)
ax.bar(x + w, [LEAN_PUBLISHED[g] for g in groups], w, label="published GFP + XGBoost (IBM paper)", color=GREY)
for xi, vals in zip([x - w, x, x + w], [ours, retro_v, [LEAN_PUBLISHED[g] for g in groups]]):
    for xx, v in zip(xi, vals):
        if not np.isnan(v): ax.text(xx, v + 1, f"{v:.1f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(groups); ax.set_ylabel("minority-class F1"); ax.set_ylim(0, 100); ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.set_title("F1 across dataset sizes (published numbers use their own pipeline)")
plt.tight_layout(); plt.show()

## 9. Laundering that isn't in any listed scheme

About 38% of the laundering transactions in `HI-Small` don't appear in `*_Patterns.txt`, so they belong to none of the 8 listed scheme types. They matter because the model catches almost none of them (section 7). A first look at what they are:

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

if pat_file.exists():
    full = add_features(tx).merge(pat_u[["ts", "from_acct", "to_acct", "fmt_raw"]].rename(columns={"fmt_raw": "fmt"}).assign(listed=1),
                                  on=["ts", "from_acct", "to_acct", "fmt"], how="left")
    full["group"] = np.where(full.label == 0, "normal", np.where(full.listed == 1, "laundering, listed", "laundering, unlisted"))
    full["same_bank"] = (full.from_bank == full.to_bank).astype(int); full["is_ach"] = (full.fmt == "ACH").astype(int)
    prof = full.groupby("group").agg(rows=("label", "size"), median_amount=("amt_paid", "median"), first_time_pair=("is_first_pair", "mean"),
                                     ach_share=("is_ach", "mean"), same_bank_share=("same_bank", "mean"))
    un = full[full.group == "laundering, unlisted"]
    print(f"{int(full.label.sum()):,} laundering transactions in total; {len(un):,} ({len(un) / full.label.sum():.0%}) are not in any listed scheme")
    listed_accts = set(pat_u.from_acct) | set(pat_u.to_acct)
    touch = lambda d_: (d_.from_acct.isin(listed_accts) | d_.to_acct.isin(listed_accts)).mean()   # noqa: E731
    print(f"share touching an account used in a listed scheme: unlisted {touch(un):.1%}   normal {touch(full[full.label == 0]):.1%}")
    codes = pd.factorize(pd.concat([un.from_acct, un.to_acct]))[0]; m_ = len(un)
    _, comp = connected_components(coo_matrix((np.ones(m_), (codes[:m_], codes[m_:])), shape=(codes.max() + 1,) * 2), directed=False)
    sz = pd.Series(comp[codes[:m_]]).map(pd.Series(comp[codes[:m_]]).value_counts())
    groups_ = pd.cut(sz, [0, 1, 2, 5, 20, 10**9], labels=["alone", "2", "3-5", "6-20", ">20"]).value_counts().reindex(["alone", "2", "3-5", "6-20", ">20"])
    print("unlisted rows by the size of the connected group they sit in (rows):", groups_.to_dict())
    mix = pd.crosstab(full.group, full.fmt, normalize="index") * 100
    ax = mix.loc[["normal", "laundering, listed", "laundering, unlisted"]].plot.barh(stacked=True, figsize=(10, 3.4), colormap="tab20")
    ax.set_xlabel("% of the group's transactions"); ax.set_ylabel(""); ax.set_title("Payment format mix"); ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.show()
    display(prof.round(3))

- **A different population from the listed schemes.** Only about 2% of the unlisted rows involve an account used in a listed scheme, the same rate as ordinary transactions.
- **Mostly stand-alone.** Most are single transactions that share no account with any other unlisted row; the rest sit in a few larger groups that may be unlisted schemes.
- **They look more like normal traffic than the listed schemes do:** smaller amounts, many cheque, card and cash payments, but still mostly ACH and between different banks. This is why the model, which relies on account history, rarely flags them.
- **Not investigated:** why the patterns file omits them, and what separates them from ordinary ACH payments.

## 10. Checks on the features

In [ ]:
# (a) Causality: delete every row after the cut and recompute; features of the kept rows must not change.
cut = int(.8 * len(tx))
full_ = add_features(tx).iloc[:cut]
part_ = add_features(tx.iloc[:cut].reset_index(drop=True))
bad = [c for c in NEW_FEATURES if not np.allclose(full_[c].to_numpy(float), part_[c].to_numpy(float), equal_nan=True)]
print("features that change when the future is deleted:", bad or "none")

# (b) Is the order of rows inside a minute informative? (see the caveat at the top)
g = tx.groupby("ts").cumcount() / tx.groupby("ts").ts.transform("size")
print(f"laundering rate in first third of a minute: {tx.label[g < 1/3].mean():.4%}   last third: {tx.label[g >= 2/3].mean():.4%}")

## Notes

- **Reference numbers** (minority-class F1, HI-Small, from the IBM paper's table): GIN 28.7, PNA 56.8, LightGBM on graph features 62.9, XGBoost on graph features 63.2. The paper's baselines also process transactions in the order listed in the file, and its graph models see a whole snapshot, so none of them uses only past transactions either.
- **What the 20 features are:** pair history (4), sender history (4), receiver history (2), velocity over 1 hour / 24 hours / 7 days (10: sender transaction counts, new payees the sender added, receiver transaction counts, plus one new-payee share).
- **Where it fails:** cheque, cash, card and bitcoin payments (never caught), senders with no activity in the previous 24 hours, and very busy senders.
- **Charts are descriptive.** The laundering-rate and recall bars show associations in this synthetic data, not causes.
- **Data and citation:** Altman et al., *Realistic Synthetic Financial Transactions for Anti-Money Laundering Models*, NeurIPS 2023 (Kaggle: `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`). Check the dataset's licence on Kaggle before redistributing any of its files.

In [ ]:
print(f"Whole notebook, from the first cell to this one: {_fmt_time(time.time() - _NOTEBOOK_START)}")

Fun Fact: "These features plus a transformer match the best published result”